# ExaHyPE 2 Tutorials - Part 5: 3D Elastic Waves and a Taste of HPC

The final tutorial steps up to **three dimensions** and to a system with **nine unknowns**: the linear elastic wave equations, the workhorse of computational seismology. An isotropic *explosion* at the centre of a cube radiates an outgoing spherical wave. We visualise it by **slicing the 3D cube** interactively.

A 3D high-order ADER-DG solver is also a genuinely **HPC-scale** workload with lots of floating-point work per cell. This part doubles as a tour of how ExaHyPE uses parallel hardware. At the end we measure its **OpenMP strong scaling** directly.

The implementation is in **`ADERDGSolver.cpp`**; open it alongside this notebook.

## The Elastic Wave Equations

We use the first-order **velocity-stress** form. The unknowns are the velocity $\mathbf{v} = (v_x, v_y, v_z)$ and the symmetric stress tensor $\boldsymbol{\sigma} = (\sigma_{xx}, \sigma_{yy}, \sigma_{zz}, \sigma_{xy}, \sigma_{xz}, \sigma_{yz})$ - nine fields in total:

$$
\rho\,\partial_t v_i = \partial_j \sigma_{ij}, \qquad
\partial_t \sigma_{ij} = \lambda\, \delta_{ij}\, \partial_k v_k
+ \mu\left(\partial_i v_j + \partial_j v_i\right).
$$

Here $\rho$ is the density and $\lambda, \mu$ are the Lamé parameters. The medium supports two wave types: fast **P-waves** at $c_p = \sqrt{(\lambda + 2\mu)/\rho}$ and slower **S-waves** at $c_s = \sqrt{\mu/\rho}$. We pick $\rho = 1$, $c_p = 2$, $c_s = 1$. An isotropic explosion (a Gaussian bump on the diagonal stresses) excites only P-waves. We expect a clean expanding spherical shell.

## Importing the API

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../../api/"))
import exahype2

## Defining the Project

A $3 \times 3 \times 3$ **cube** centred on the origin with non-periodic walls. The only change from the 2D tutorials is `dimensions = 3` and the three-component `size`/`offset`/`periodic_bc`.

In [ ]:
my_project = exahype2.Project(
    namespace    = ["tutorials", "exahype2", "elastic"],
    project_name = "Elastic",
    executable   = "Elastic",
    dimensions   = 3,
    size         = [ 3.0,  3.0,  3.0],
    offset       = [-1.5, -1.5, -1.5],
    periodic_bc  = [False, False, False],
)

## Defining the Solver

An ADER-DG solver of order 4. A dictionary lists the unknowns. The velocity `v` has multiplicity 3 and the stress `sigma` multiplicity 6. We address them by name in C++ (`Shortcuts::v + 0..2`, `Shortcuts::sigma + 0..5`).

In [ ]:
aderdg_solver = exahype2.dg.ader.Solver(
    name                 = "ADERDGSolver",
    order                = 4,
    unknowns             = {"v": 3, "sigma": 6},
    auxiliary_variables  = {},
    time_stepping        = exahype2.TimeStepping.Adaptive(0.9),
    linear               = True,
)

We implement the **flux**, the **maximum eigenvalue** ($c_p$) and the initial and boundary conditions. The elastic system is linear. `linear = True` above switches on the optimised linear ADER-DG kernels.

In [ ]:
aderdg_solver.set_pde_terms(
    initial_condition   = True,
    boundary_conditions = True,
    flux                = True,
    max_eigenvalue      = True,
)

my_project.add_solver(aderdg_solver)

## Building the Project

Generate and compile. In 3D at order 4 each cell stores $(4+1)^3 = 125$ degrees of freedom per unknown. The build and the run are heavier than in the 2D tutorials.

In [ ]:
my_project.output.makefile.add_cpp_file("ADERDGSolver.cpp")
my_project.generate_and_build()

## The Implementation File

Open **`ADERDGSolver.cpp`**. `initialCondition` sets the Gaussian explosion (equal diagonal stresses, at rest); `boundaryConditions` gives a quiescent exterior to reflect off; `maxEigenvalue` returns the P-wave speed $c_p$; and `flux` is the velocity-stress flux, written out per direction with the Lamé parameters recovered from $\rho, c_p, c_s$.

## Running the Simulation

In [ ]:
import os, subprocess

# Runtime parameters - edit these freely, no rebuild needed.
end_time  = 0.5     # -et : physical end time
snapshots = 10      # -ns : number of output snapshots (drives the animation)
level     = 2       # -l  : uniform mesh refinement level (k^level cells per axis, k = 3 on the Peano curve, 2 on the Hilbert curve)

# Run with 4 OpenMP threads (the notebook equivalent of: export OMP_NUM_THREADS=4)
os.environ["OMP_NUM_THREADS"] = "4"

subprocess.run(
    ["./Elastic", "-et", str(end_time), "-ns", str(snapshots), "-l", str(level)],
    check=True,
)

## Visualising in 3D

A 3D field is best explored **slice by slice**. `show_solution_3D` cuts axis-aligned 2D planes through the cube: pick the slice axis with the **slice ⊥** dropdown (x/y/z), move the **slice** slider through the volume, pick a field, and animate with the **time** slider / **Play**. Picking a slice (or field/axis) renders all of that slice's time frames in parallel across cores and caches them during a brief *Prefetching N/M*. The **time** slider and **Play** then run smoothly.

The expanding spherical P-wave appears in the mid-plane slice as **concentric rings** - the cross-section of the spherical shell. Move the slice slider towards a face of the cube and the rings shrink, exactly as cutting a sphere off-centre should.

*(The reference pictures in the documentation were rendered with `level = 3` - raise it in the run cell and re-run for the same crispness; the run then takes a few minutes.)*

In [ ]:
import importlib, os, sys
sys.path.insert(0, os.path.abspath("../../"))

from postprocessing import render_solution
importlib.reload(render_solution)

# Interactive slice viewer: pick the slice axis (x/y/z), move the slice slider
# through the cube, pick a field and animate with the time slider / Play.
render_solution.show_solution_3D("ADERDGSolver", field="sigma")

## Going Parallel - HPC in ExaHyPE 2

A 3D high-order ADER-DG solver is exactly the compute-bound workload HPC exists for. ExaHyPE exposes parallelism through two control surfaces: **distributed memory** (`mpirun -n N ./Elastic ...`) and **threads** (`OMP_NUM_THREADS`). Neither needs a code change. The documentation page (*Part 5 - 3D Elastic Waves and a Taste of HPC*, section *A Taste of HPC*) explains both and the banner lines they print.

The cell below measures the second one, **OpenMP strong scaling**: the same problem on 1, 2 and 4 threads of one process with plotting switched off (`-ns 0`, to time pure compute), reading the `Average Time Step` the solver reports.

In [ ]:
import os, re, subprocess

level = 2

def time_per_step(threads):
    """Run the solver with `threads` OpenMP threads; return the average time/step (s)."""
    env = dict(os.environ, OMP_NUM_THREADS=str(threads))
    log = subprocess.run(
        ["./Elastic", "-et", "0.3", "-ns", "0", "-l", str(level)],
        env=env, capture_output=True, text=True, check=True,
    ).stdout
    return float(re.search(r"Average Time Step\s+([\d.eE+-]+)\s*s", log).group(1))

thread_counts = [1, 2, 4]
timings = [time_per_step(n) for n in thread_counts]
for n, t in zip(thread_counts, timings):
    print(f"{n} thread(s): {t * 1e3:6.1f} ms/step   speedup x{timings[0] / t:.2f}")

The plot below compares the measured speedup with the ideal linear line. The gap widens as you add threads at a fixed problem size. On one node, threads scale better than as many MPI ranks with one thread each. Every rank copies the faces on the boundary of its subdomain into messages and waits in MPI for the faces of its neighbours in every time step. The threads of one process read these faces in place.

In [ ]:
import matplotlib.pyplot as plt

speedup = [timings[0] / t for t in timings]
plt.figure(figsize=(5, 4))
plt.plot(thread_counts, speedup, "o-", label="measured")
plt.plot(thread_counts, thread_counts, "k--", label="ideal (linear)")
plt.xlabel("OpenMP threads")
plt.ylabel("speedup")
plt.title("OpenMP strong scaling")
plt.legend()
plt.show()

## Exercise - Launch a Shear Wave

The isotropic explosion sets the three **diagonal** stresses equal. That is a pure
volume change. It radiates only the fast **P-wave**, the clean spherical shell (a
ring in any slice). A **shear** source, a single *off-diagonal* stress, launches
the slower **S-wave** ($c_s = 1$, half the P-wave speed). Its radiation forms
a four-lobed **quadrupole**.

**Your task.** In `ADERDGSolver.cpp`, inside `initialCondition`, replace the three
diagonal-stress lines

```cpp
Q[Shortcuts::sigma + 0] = bump;   // sigma_xx
Q[Shortcuts::sigma + 1] = bump;   // sigma_yy
Q[Shortcuts::sigma + 2] = bump;   // sigma_zz
```

with a single off-diagonal (shear) component:

```cpp
Q[Shortcuts::sigma + 3] = bump;   // sigma_xy (shear) -> launches an S-wave
```

Then re-run the **build** cell and the **run** cell, and run the viewer below with
the slice axis on **z** at the middle slice position: in place of the P-wave's
expanding ring you will see the four lobes of the S-wave.

In [ ]:
# After editing the IC to a shear source and re-running build + run, view the
# velocity field sliced along z at the mid-plane: the four-lobed S-wave
# quadrupole replaces the P-wave's ring.
render_solution.show_solution_3D("ADERDGSolver", field="v")

## Experiment

Two tweaks you can make above and re-run, no rebuild: raise `level` to 3 for a heavier (and usually better-scaling) run, and add `8` to `thread_counts` if your machine has the cores.

For the rest - going hybrid with several threads per rank, and raising the solver `order` - and a closing word on the tutorial series, see the **Experiment** section of the documentation page, *Part 5 - 3D Elastic Waves and a Taste of HPC*.